In [1]:
import os
import json
from tqdm import tqdm

def read_json_file(path: str) -> dict | list:
    with open(path, 'r') as f:
        return json.load(f)

def write_json_file(data: dict | list, path: str) -> None:
    with open(path, 'w') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

In [2]:
raw_data = read_json_file("./data/raw_data/QueryRefiner.json")
for d in raw_data:
    d["input_data"] = json.loads(d["input_data"])
    d["output_data"] = json.loads(d["output_data"])

print(len(raw_data))
raw_data[0]

2581


{'trace_id': '47a28a4cf846642dd73af5d412aa3133',
 'chain_name': 'QueryRefiner',
 'input_data': {'user_input': 'سلام ب نظرت میتونی کمکی بکنی؟\nدرخواست کارت دادم بیشتراز ده روزه هنوز ب دستم نرسیده',
  'history': '',
  'title': 'راهنمای جامع اپلیکیشن بانکت: خدمات نوین بانکی و سرویس\u200cهای ارزش افزوده',
  'language': 'PERSIAN'},
 'output_data': {'output': 'سلام. به نظرت میتونی کمکی بکنی؟ من درخواست صدور کارت بانکت دادم و بیش از ده روزه که هنوز به دستم نرسیده.'},
 'metadata': {'completion_tokens': 48,
  'prompt_tokens': 1534,
  'total_cost': 0.00018209999999999998},
 'collection_info_version': 173}

In [6]:
def format_input(input_data: dict) -> str:
    text = f"""<QueryRefiner>\
<title>{input_data['title']}</title>\
<history>{input_data['history']}</history>\
<user_input>{input_data['user_input']}</user_input>\
<language>{input_data['language']}</language>\
</QueryRefiner>"""    
    return text

In [22]:
def convert_to_alpaca_format(raw_data: list) -> list:
    final_data = []
    for item in tqdm(raw_data):
        final_data.append({
            "input": format_input(item["input_data"]),
            "output": f'{item["output_data"]}'
        })
    return final_data

In [23]:
alpaca_data = convert_to_alpaca_format(raw_data)
write_json_file(alpaca_data, "./data/processed_data/QueryRefiner_Alpaca.json")
write_json_file(alpaca_data[:500], "./data/processed_data/QueryRefiner_Alpaca500.json")

100%|██████████| 2581/2581 [00:00<00:00, 89529.82it/s]

In [24]:
SYSTEM_PROMPT = """\
Analyze the <user_input> alongside the <history> and <title> to identify and resolve any ambiguous references or pronouns.\
 Replace these references with specific subjects from the context to create a standalone query,\
 or leave the input as-is if it is already complete.\
 Translate the final output into <language>.\
 Preserve the original intent and format of the user's input in the refined response.\
"""

In [25]:
def convert_to_OpenAI_format(raw_data: list) -> list:
    final_data = []
    for item in tqdm(raw_data):
        final_data.append({
            "messages": [
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": format_input(item["input_data"])
                },
                {
                    "role": "assistant",
                    "content": f'{item["output_data"]}'
                }
            ]
        })
    return final_data

In [26]:
openai_data = convert_to_OpenAI_format(raw_data)
write_json_file(openai_data, "./data/processed_data/QueryRefiner_OpenAI.json")
write_json_file(openai_data[:500], "./data/processed_data/QueryRefiner_OpenAI500.json")

100%|██████████| 2581/2581 [00:00<00:00, 108232.26it/s]


In [ ]:
print(len(d1), len(d2), len(d3))